# Notebook 3: L298N Motor Control

This is the first notebook where your code moves an actual, physical robot.
Everything before this (LED, RGB LED, button) was contained on a breadboard
with no moving parts. A DC motor is different: once it's spinning, it keeps
spinning — under its own momentum and the force it's generating — until
something explicitly stops it.

> ## SAFETY WARNING — READ BEFORE RUNNING ANY CELL BELOW
>
> **This notebook moves a real motor.** Before you run any code cell past
> this point:
>
> - **Lift the robot chassis off the ground/table**, or otherwise make sure
>   all four wheels can spin completely freely, with nothing they can catch
>   on, drive into, or pull (power cables, USB cables, your hand).
> - Keep the robot lifted/blocked for the **entire notebook**, not just the
>   first cell — every section below runs a motor again.
> - Know where **`stop(motors)`** is before you start, and don't hesitate to
>   run it (or interrupt the kernel) if anything looks wrong.
>
> This is not a formality — a wheeled robot with its motors suddenly
> running on a table or floor can drive itself off the edge, tangle its own
> wiring, or drive into something (or someone) before you can react.

## What you'll learn

1. **Why a Raspberry Pi can't drive a motor directly** off its GPIO pins,
   and what an **H-bridge motor driver** (the L298N) is for.
2. How to wire the L298N to the Pi and to the motors/motor power supply.
3. Controlling the **left** motor and the **right** motor independently:
   forward, backward, stop.
4. Combining both motors into **robot-level** movement: driving straight,
   and pivot-turning in place.
5. **PWM speed control** via `set_speed()` — the same duty-cycle idea from
   Notebook 2's RGB LED, now controlling how fast (not how bright)
   something runs.

As in every notebook so far, we're using an already-implemented, already
electrically-verified-on-the-real-Pi project module: `src/hardware/motor.py`
— every direction combination, genuine PWM on the speed-control pins, and
the safety-critical behavior of `stop()` were all confirmed on the real Pi
before this notebook was written. This notebook teaches that module's
functions one at a time, in the order the module itself was designed to be
taught in.

## Why can't the Pi drive a motor directly?

Every GPIO pin you've used so far — the LED, the RGB LED's three channels,
even reading the button — deals with tiny amounts of current: an LED
typically draws somewhere around 10-20 milliamps, comfortably within what a
single GPIO pin can safely source. A Raspberry Pi GPIO pin is only rated to
source a small amount of current (on the order of 16mA per pin, with a
modest total limit across all pins combined) at 3.3V.

A DC motor is a completely different scale of load. Spinning a motor —
especially one under any mechanical load, like turning wheels against
friction — can draw many hundreds of milliamps to a few amps, often at a
higher voltage than the Pi's 3.3V logic (this project's motor supply
voltage is a separate battery pack from the Pi's own power — see
`HARDWARE.md`). Connecting a motor directly to a GPIO pin wouldn't just
fail to spin it properly — it would draw far more current than the pin can
supply, which can damage the GPIO pin or the whole board.

There's a second problem beyond current: motors need to run in **two
directions** (forward and backward), which means reversing the polarity of
the voltage across them. A GPIO pin can only be HIGH or LOW — it has no
built-in way to "reverse" its own output.

Both problems are solved by putting a separate, purpose-built component
between the Pi and the motors: an **H-bridge motor driver**. This project
uses the **L298N**, a common, inexpensive dual H-bridge driver board. It:

- takes **logic-level signals** from the Pi's GPIO pins (safe, low-current,
  3.3V) as *direction* and *speed* commands, and
- separately draws real motor current from its **own, more powerful power
  supply** (the motor battery, not the Pi), switching that higher-current
  power through internal transistors to the motors in whichever direction
  the logic signals requested.

In other words: the Pi never directly powers the motors at all. It only
ever sends small, safe control signals to the L298N, and the L298N does the
actual work of driving current through the motors. This is exactly the same
reason the RGB LED and plain LED needed current-limiting resistors, just
scaled up — the Pi's GPIO pins are signal pins, not power sources, for
anything beyond the smallest loads.

## Wiring the L298N

**Do this fully before running any code below**, and re-read the safety
warning at the top of this notebook before you power anything on.

### Logic/control connections (Pi → L298N)

| L298N pin | Connect to | BCM pin | Physical pin |
|---|---|---|---|
| ENA (left motor speed) | GPIO12 | 12 | physical pin 32 |
| IN1 (left motor direction) | GPIO5 | 5 | physical pin 29 |
| IN2 (left motor direction) | GPIO6 | 6 | physical pin 31 |
| ENB (right motor speed) | GPIO13 | 13 | physical pin 33 |
| IN3 (right motor direction) | GPIO19 | 19 | physical pin 35 |
| IN4 (right motor direction) | GPIO26 | 26 | physical pin 37 |

ENA/ENB are the L298N's PWM speed inputs, wired to the Pi's two hardware
PWM-capable channels — this is why speed control on this project's two
motors goes through GPIO12/GPIO13 specifically rather than any other pins.

### Power connections

- The L298N's motor power input terminals connect to a **separate motor
  battery pack**, not the Pi's own power supply. (Exact battery voltage is
  project-specific — see `HARDWARE.md`; the L298N tolerates a range of
  input voltages, but check its rating against your battery before
  connecting.)
- The L298N's two motor output pairs (OUT1/OUT2, OUT3/OUT4) connect to the
  left and right DC motors respectively.
- **Do not connect the L298N's 5V output pin to the Pi.** Some L298N boards
  have an onboard 5V regulator that can output 5V from the motor supply —
  useful for powering other 5V logic, but never intentionally back-feed
  that into the Pi, and never assume the Pi's own 5V rail can power the
  motors (SAFETY.md already documents both directions of this rule).

### Common ground — required, easy to miss

The L298N's **GND** must be connected to **both**:

- a **GND pin on the Pi** (so the Pi's HIGH/LOW logic signals on
  IN1-4/ENA/ENB have a shared 0V reference the L298N can actually
  interpret), **and**
- the **motor battery's GND** (so the L298N's power stage and its logic
  stage agree on what "0V" means).

Without a shared ground between the Pi and the motor supply, the L298N
can't reliably tell what the Pi's HIGH/LOW signals mean relative to its own
power rail — direction and speed control can behave erratically or not at
all, even though every individual wire looks "connected." This is the same
"grounds are tied together" note from `HARDWARE.md`'s bill of materials —
worth actually double-checking with a multimeter or continuity check if
available, since a missing ground connection is a notoriously easy thing to
visually miss on a breadboard.

## Setup

### Explanation

Add `src/` to the path (same as every previous notebook) and import the
motor module's functions. `get_motors()` creates the `Motors` bundle — four
GPIO objects total (direction pins for each motor, plus a separate PWM
object for each motor's enable pin) — and leaves both motors' speed at 0%,
so **nothing moves as a result of running this cell**, only after you call
one of the movement functions below.

In [ ]:
import sys
import time
sys.path.insert(0, '../src')

from hardware.motor import (
    get_motors,
    left_forward, left_backward, left_stop,
    right_forward, right_backward, right_stop,
    forward, backward, stop,
    left, right,
    set_speed,
    cleanup,
)

motors = get_motors()
print("Motors ready. Both enable pins at 0% - nothing should be moving.")


### Expected result

`Motors ready. Both enable pins at 0% - nothing should be moving.` printed,
no error.

### Physical result

Nothing — no motor should move as a result of this cell. If a wheel *is*
spinning right now, stop and recheck your wiring before continuing (most
likely cause: a direction pin wired to the wrong GPIO, or a miswired
enable/PWM connection).

## Left motor

> ## SAFETY WARNING — READ BEFORE RUNNING ANY CELL BELOW
>
> **This notebook moves a real motor.** Before you run any code cell past
> this point:
>
> - **Lift the robot chassis off the ground/table**, or otherwise make sure
>   all four wheels can spin completely freely, with nothing they can catch
>   on, drive into, or pull (power cables, USB cables, your hand).
> - Keep the robot lifted/blocked for the **entire notebook**, not just the
>   first cell — every section below runs a motor again.
> - Know where **`stop(motors)`** is before you start, and don't hesitate to
>   run it (or interrupt the kernel) if anything looks wrong.
>
> This is not a formality — a wheeled robot with its motors suddenly
> running on a table or floor can drive itself off the edge, tangle its own
> wiring, or drive into something (or someone) before you can react.

The next cell is the first one that actually spins a motor. Confirm — again
— that the robot is lifted or the wheels are otherwise free to spin before
you run it.

### Explanation

`left_forward(motors)` drives the left motor forward at the module's
default speed (50%). It sets the left enable/PWM pin to 50% duty cycle and
tells the left motor's direction pins to spin forward — and then, just like
every gpiozero movement call, **returns immediately**. The motor keeps
spinning after the cell finishes running, until something explicitly stops
it — which is exactly what the very next cell does. Don't skip ahead; run
this cell, observe briefly, then move straight to the stop cell below it.

In [ ]:
left_forward(motors)


### Expected result

No output, no error.

### Physical result

If the robot is lifted off the ground, the **left wheel(s)** should spin.
The right wheel(s) should stay still — only the left motor was commanded.
If nothing spins, or the wrong wheel spins, see the troubleshooting note
below before assuming the code is wrong.

Run the next cell now to stop it — don't leave this spinning.

### Explanation

Stop the left motor immediately. `left_stop(motors)` cuts the left motor's
direction signal **and** zeroes its PWM speed — both, not just one — so the
motor is never left "armed" at a nonzero speed after stopping.

In [ ]:
left_stop(motors)


### Expected result

No output, no error.

### Physical result

The left wheel(s) should stop spinning.

### Explanation

Now the reverse direction: `left_backward(motors)` at the same default 50%
speed. As before, it returns immediately and keeps running until stopped —
run the stop cell right after observing it.

In [ ]:
left_backward(motors)


### Expected result

No output, no error.

### Physical result

The left wheel(s) should spin in the **opposite** direction from the
`left_forward()` cell above.

**Troubleshooting — if direction seems reversed or nothing spins:** this is
a real, expected debugging scenario, not necessarily a code problem.
`left_forward()`/`left_backward()` assume the motor's two leads are wired to
the L298N's OUT1/OUT2 terminals in a particular, consistent polarity. If
forward and backward come out swapped (or a turn later in this notebook
comes out mirrored), the fix is to **physically swap that motor's two wire
leads at the L298N terminal** — not to edit `motor.py`. If nothing spins at
all in either direction, recheck ENA/IN1/IN2 wiring and the motor power
connections first.

In [ ]:
left_stop(motors)


### Expected result

No output, no error.

### Physical result

The left wheel(s) should stop. You've now exercised `left_forward()`,
`left_backward()`, and `left_stop()` (twice) independently — this is the
same "test one function at a time" verification QA already did
electrically; you're now confirming it mechanically.

## Right motor

Same three functions, mirrored for the right motor: `right_forward()`,
`right_backward()`, `right_stop()`. The robot should still be lifted/free-
spinning from the safety warning above — if you set it down between
sections, lift it again now before continuing.

### Explanation

`right_forward(motors)` — same idea as `left_forward()`, but only the right
motor. Stop it with the next cell right after observing.

In [ ]:
right_forward(motors)


### Expected result

No output, no error.

### Physical result

The **right** wheel(s) should spin; the left should stay still.

In [ ]:
right_stop(motors)


### Expected result

No output, no error.

### Physical result

The right wheel(s) should stop.

### Explanation

`right_backward(motors)` — the right motor's reverse direction.

In [ ]:
right_backward(motors)


### Expected result

No output, no error.

### Physical result

The right wheel(s) should spin opposite to the `right_forward()` cell
above. Same polarity troubleshooting note as the left motor applies here if
it looks wrong — swap that motor's leads at the L298N terminal, don't edit
code.

In [ ]:
right_stop(motors)


### Expected result

No output, no error.

### Physical result

The right wheel(s) should stop.

## Combined robot-level movement

> ## SAFETY WARNING — READ BEFORE RUNNING ANY CELL BELOW
>
> **This notebook moves a real motor.** Before you run any code cell past
> this point:
>
> - **Lift the robot chassis off the ground/table**, or otherwise make sure
>   all four wheels can spin completely freely, with nothing they can catch
>   on, drive into, or pull (power cables, USB cables, your hand).
> - Keep the robot lifted/blocked for the **entire notebook**, not just the
>   first cell — every section below runs a motor again.
> - Know where **`stop(motors)`** is before you start, and don't hesitate to
>   run it (or interrupt the kernel) if anything looks wrong.
>
> This is not a formality — a wheeled robot with its motors suddenly
> running on a table or floor can drive itself off the edge, tangle its own
> wiring, or drive into something (or someone) before you can react.

Now that both motors are individually confirmed working, combine them.
`forward()`, `backward()`, `left()`, and `right()` are built directly from
the per-motor functions you just tested — `forward()` is simply
`left_forward()` + `right_forward()` together, for example — so nothing
new is happening at the hardware level, just both motors commanded
together instead of one at a time.

### Explanation

`forward(motors)` drives both motors forward together, so the robot should
move in a straight line (assuming both motors are wired with matching
polarity — see the troubleshooting note above if it curves instead).

In [ ]:
forward(motors)


### Expected result

No output, no error.

### Physical result

If lifted, both wheels spin forward together. If actually resting on the
ground/a surface with room to move, the robot should roll forward in a
reasonably straight line — small drift is normal at this stage (motors
rarely spin at exactly matched speeds even at the "same" duty cycle) and is
addressed in this notebook's exercises.

In [ ]:
stop(motors)


### Expected result

No output, no error.

### Physical result

Both wheels stop. `stop(motors)` calls both `left_stop()` and
`right_stop()` — direction and speed zeroed on both motors.

### Explanation

`backward(motors)` — both motors in reverse together.

In [ ]:
backward(motors)


### Expected result

No output, no error.

### Physical result

Both wheels spin backward together (or the robot rolls backward, if set down with room to move).

In [ ]:
stop(motors)


### Expected result

No output, no error.

### Physical result

Both wheels stop.

### Explanation

`left(motors)` is a **pivot turn** — it spins the robot in place by driving
the left motor backward and the right motor forward at the same time
(rather than steering like a car). This only makes sense with all wheels
free to spin, or the robot lifted — a pivot turn on the ground needs
enough grip/space to actually rotate in place.

In [ ]:
left(motors)


### Expected result

No output, no error.

### Physical result

Left wheel(s) spin backward, right wheel(s) spin forward at the same time
— if lifted, you'll see the two sides spinning in opposite directions; if
resting with room to move, the robot should rotate in place toward the
left. If it rotates the wrong way, that's the same per-motor polarity issue
covered earlier — check both motors' lead polarity, since a pivot turn
depends on both motors agreeing on which physical direction is "forward."

In [ ]:
stop(motors)


### Expected result

No output, no error.

### Physical result

Both wheels stop.

### Explanation

`right(motors)` — the mirror pivot: left motor forward, right motor
backward.

In [ ]:
right(motors)


### Expected result

No output, no error.

### Physical result

The robot pivots the opposite way from `left(motors)` above.

In [ ]:
stop(motors)


### Expected result

No output, no error.

### Physical result

Both wheels stop.

## Speed control with `set_speed()`

Recall from Notebook 2: PWM (pulse width modulation) makes something look
"partway on" by switching rapidly between fully on and fully off, where the
**duty cycle** — the percentage of time spent HIGH — determines the
perceived effect. There, duty cycle controlled *brightness*. Here, the
exact same mechanism, applied to the L298N's ENA/ENB pins, controls
*speed*: a higher duty cycle delivers more average power to the motor,
which spins faster.

Every movement function you've used so far (`left_forward()`, `forward()`,
etc.) already takes an optional `speed_percent` argument (0-100, defaulting
to 50) — so you've been using PWM speed control the whole time, just at a
fixed default. `set_speed(motors, left_pct, right_pct)` lets you change
each side's speed **independently**, at any time, without changing
direction — useful both for deliberately turning while still moving
(faster on one side than the other) and, as this notebook's exercises will
use it for, correcting a robot that doesn't drive perfectly straight.

### Explanation

Drive forward at a low, cautious speed pair first — 30% on each side —
using `set_speed()` before starting movement. Since `set_speed()` only
changes the PWM duty cycle and has no effect on direction, calling it before
any direction is set has no visible effect by itself; the actual movement
starts when `forward()` runs next.

In [ ]:
set_speed(motors, 30, 30)
forward(motors)


### Expected result

No output, no error.

### Physical result

Both wheels spin forward, noticeably slower than the earlier `forward()`
cell (which used the 50% default).

In [ ]:
stop(motors)


### Expected result

No output, no error.

### Physical result

Both wheels stop.

### Explanation

Now try an **uneven** speed pair — 80% left, 30% right — while moving
forward. Because both motors are still commanded to spin *forward*
(direction unchanged), this doesn't pivot the robot in place like
`left()`/`right()` did — it curves while moving, since one side is turning
much faster than the other. This is the same idea steering-by-differential-
speed uses on a two-motor robot, and is what one of this notebook's
exercises builds on.

In [ ]:
set_speed(motors, 80, 30)
forward(motors)
time.sleep(1)
stop(motors)


### Expected result

No output, no error. This cell pauses for about 1 second before stopping on
its own (the first cell in this notebook to include its own timed stop,
rather than relying on a separate stop cell — both patterns are fine, as
long as a stop always happens).

### Physical result

If lifted: the left wheel(s) visibly spin much faster than the right. If
resting with room to move: the robot should curve/arc toward the slower
(right) side rather than driving straight, for about a second, then stop.

## Cleanup

### Explanation

`cleanup(motors)` stops both motors (direction and speed both zeroed, same
as `stop()`) and then releases all four underlying GPIO objects. Run this
when you're done experimenting so the pins are cleanly available again —
for another notebook, another kernel session, or a re-run of `get_motors()`
from the top of this notebook.

In [ ]:
cleanup(motors)
print("Motors stopped and all GPIO released.")


### Expected result

`Motors stopped and all GPIO released.` printed, no error.

### Physical result

Both motors should already have been stopped before this cell (every
movement cell above was paired with its own stop), so there should be no
visible change here beyond the pins being released — but if anything were
still moving for any reason, this cell stops it too, as a final safety
net.

## Recap

- A Raspberry Pi GPIO pin can safely source only a small amount of current
  at 3.3V — nowhere near enough to drive a motor directly, and motors also
  need their direction reversed, which a single GPIO pin can't do on its
  own. The **L298N H-bridge driver** solves both problems: it takes small
  logic-level signals from the Pi and switches a separate, higher-power
  motor supply through the motors accordingly.
- `src/hardware/motor.py` wires this up as: `Motor(forward=IN_a,
  backward=IN_b, pwm=False)` per side for **direction only**, plus a
  separate `PWMOutputDevice` per side on ENA/ENB for **real PWM speed
  control** — deliberately not using `gpiozero.Motor`'s built-in `enable=`
  parameter, because (per the module's own source-reading) that parameter
  only ever holds the enable pin statically HIGH rather than pulsing it,
  which would leave ENA/ENB never actually PWM'd. This is worth
  remembering as a general lesson: a library's documented parameter name
  doesn't always do what it sounds like it does — check the source when it
  matters.
- Per-motor primitives (`left_forward`/`left_backward`/`left_stop`,
  `right_forward`/`right_backward`/`right_stop`) combine into robot-level
  functions (`forward`, `backward`, `left`, `right`, `stop`) that are
  nothing more than both per-motor functions called together.
- `set_speed(motors, left_pct, right_pct)` changes PWM duty cycle
  independently per side, without touching direction — the same duty-cycle
  mechanism from Notebook 2, now controlling speed instead of brightness.
- Every movement function has a matching `stop()`, and every movement cell
  in this notebook was paired with an explicit stop — that discipline
  matters more here than in any previous notebook, because a motor, unlike
  an LED, keeps physically moving (and can keep causing consequences) after
  your code stops actively driving it, until something cuts power or it
  runs down under friction.

## Exercises

> ## SAFETY WARNING — READ BEFORE RUNNING ANY CELL BELOW
>
> **This notebook moves a real motor.** Before you run any code cell past
> this point:
>
> - **Lift the robot chassis off the ground/table**, or otherwise make sure
>   all four wheels can spin completely freely, with nothing they can catch
>   on, drive into, or pull (power cables, USB cables, your hand).
> - Keep the robot lifted/blocked for the **entire notebook**, not just the
>   first cell — every section below runs a motor again.
> - Know where **`stop(motors)`** is before you start, and don't hesitate to
>   run it (or interrupt the kernel) if anything looks wrong.
>
> This is not a formality — a wheeled robot with its motors suddenly
> running on a table or floor can drive itself off the edge, tangle its own
> wiring, or drive into something (or someone) before you can react.

**1. Move forward for a set duration, then stop.**
Write a cell that calls `forward(motors)`, waits a specific duration with
`time.sleep(...)`, then calls `stop(motors)` — a single self-contained cell
(like the uneven-speed example above), rather than relying on a separate
stop cell. Try a couple of different durations and speeds and see how far
the robot travels each time (if it's resting on a surface with room to
move).

**2. Trace a square.**
Using `forward()`, one of the pivot turns (`left()`/`right()`), and
`time.sleep()`, write a sequence that drives forward for a set duration,
pivots roughly 90 degrees, and repeats four times to trace out (roughly) a
square path. You will need to **empirically tune** how long the pivot
should last to approximate 90 degrees — this setup has no rotational
feedback (no encoder or gyroscope), so there's no way to measure the turn
precisely; adjust the pivot duration by trial and observation until it
looks close. Expect some drift/inaccuracy — that's normal and expected at
this level, not a sign anything is wrong.

**3. Correct a robot that doesn't drive straight.**
If your robot visibly curves during the plain `forward()` calls earlier in
this notebook, use `set_speed()` to compensate: reduce the faster side's
speed (or increase the slower side's) by small increments until `forward()`
drives as straight as you can get it. Note the speed pair that works best
for your specific robot — motor-to-motor variation is normal even with
identical model numbers, so this correction is specific to your hardware,
not something a code change alone would fix for everyone.

**4. (Optional, extra challenge) Combine a timed pattern.**
Write a short "routine" cell that chains several movements with different
speeds and durations — for example, forward slow, forward fast, pivot left,
backward, stop — using only `time.sleep()` between each step (no button or
sensor input yet; that comes in later notebooks). This previews the shape
of an autonomous behavior: a fixed sequence of INPUT-free actions, each
with its own duration, ending in a guaranteed `stop()`.